In [ ]:
from pathlib import Path

# Edit these settings for your CSV. Header names or zero-based positions work.
csv_location = Path("examples/projects.csv")
has_header = True
delimiter = ","
columns = {
    "title": "title",
    "supervisor": "supervisor",
    "description": "description",
    "tags": "tags",
    "summary": "summary",
    "tools": "tools",
    "difficulty": "difficulty",
    "ai_assistance": "ai_assistance",
    "readable_description": "readable_description",
    "draft_rank": "draft_rank",
    "topics": "topics",
}
# Omit optional mappings (or set them to None) when columns are absent.
from ranker_data import session_path
save_path = session_path(csv_location)

# Optional: let draft ranks gently influence initial preferences.
use_draft_prior = False


In [ ]:
from ranker_data import load_projects

data, project_ratings = load_projects(
    csv_location, columns, has_header=has_header, delimiter=delimiter, use_draft_prior=use_draft_prior,
)
print(f"Loaded {len(data)} projects.")


In [ ]:
width = len(data[0])
height = len(data)

print(f"width {width}, height {height}")

In [ ]:
checks = []
x = 0
for row in range(0,height,1):
    for col in range(row + 1, height):
        checks.append((row+1, col+1))

In [ ]:
len(checks)

In [ ]:
import random

random.shuffle(checks)

In [ ]:
# Preserve a running session when setup cells are rerun.
existing_ranker = globals().get("ranker")
if existing_ranker is not None and existing_ranker.projects == data and Path(existing_ranker.save_path).resolve() == Path(save_path).resolve():
    scores = existing_ranker.scores
else:
    scores = [0] * height


In [ ]:
import html
from ranker_formatting import render_readable_description
import re
import ipywidgets as widgets
from IPython.display import display
import importlib
import ranker_session as session_module
importlib.reload(session_module)
from ranker_session import load_session, save_session
import ranker_model as model_module
importlib.reload(model_module)
from ranker_model import fit_preferences, shortlist_status
import ranker_scoring as scoring_module
importlib.reload(scoring_module)
from ranker_scoring import preference_stats, ranked_indices
import ranker_schedule as schedule_module
importlib.reload(schedule_module)
from ranker_schedule import prioritise_next


def format_description(text):
    """Add structure to the original description without rewriting its wording."""
    headings = (
        r"Project description:|Deliverables:|Project approach|Project variants|"
        r"Subproject \d+:|\(Variant \d+\)[^:]*:|"
        r"Literature review:|Specification and design of the analysis approach:|"
        r"Construction of an evaluation dataset:|Implementation of the analysis and reporting system:|"
        r"Evaluation:|References on [^:]+:"
    )
    sections = re.split(rf"(?<!\w)({headings})", text)
    rendered = []
    for part in sections:
        if not part.strip():
            continue
        if re.fullmatch(headings, part):
            rendered.append(
                '<h4 style="font-size:15px;margin:20px 0 8px;font-weight:700;">'
                + html.escape(part) + '</h4>'
            )
            continue
        # Preserve existing paragraphs and separate source list items.
        blocks = re.split(r"\n\s*\n|\s{2,}|(?=\s+-\s+[A-Z])", part)
        for block in blocks:
            block = block.strip()
            if not block:
                continue
            # Long flat paragraphs are spaced every three sentences.
            sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", block)
            for start in range(0, len(sentences), 3):
                paragraph = " ".join(sentences[start:start + 3])
                rendered.append(
                    '<p style="margin:0 0 14px;white-space:pre-wrap;">'
                    + html.escape(paragraph) + '</p>'
                )
    return "".join(rendered)


def metric_badge(label, value):
    return (
        '<span style="display:inline-block;padding:6px 10px;margin:0 6px 6px 0;'
        'border:1px solid #8886;border-radius:8px;font-size:13px;">'
        f'{html.escape(label)} <b>{html.escape(str(value))}</b></span>'
    )


class ProjectRanker:
    def __init__(self, projects, comparisons, project_scores, reference_ratings, position=0, history=None, save_path=None, answer_budget=250, stability_review_after=0):
        self.projects = projects
        self.comparisons = comparisons
        self.scores = project_scores
        self.reference_ratings = reference_ratings
        self.position = position
        self.history = list(history or [])
        self.answer_budget = answer_budget
        self.stability_review_after = stability_review_after
        self.save_path = save_path
        self.save_status = widgets.HTML()
        self.status = widgets.HTML()
        self.options = widgets.HTML()
        self.ranking = widgets.HTML()
        self.choose_a = widgets.Button(description="Choose A", button_style="primary")
        self.choose_b = widgets.Button(description="Choose B", button_style="primary")
        self.back_button = widgets.Button(description="Back", tooltip="Undo the last choice or skip")
        self.skip_button = widgets.Button(description="Skip", tooltip="Next pair without changing scores")
        self.both_button = widgets.Button(description="Both", button_style="primary", tooltip="Give both projects 10 points")
        self.rank_button = widgets.Button(description="Show top 20")
        self.stop_button = widgets.Button(description="Stop")
        self.choose_a.on_click(lambda _: self.choose(0))
        self.choose_b.on_click(lambda _: self.choose(1))
        self.both_button.on_click(lambda _: self.choose(2))
        self.back_button.on_click(lambda _: self.back())
        self.skip_button.on_click(lambda _: self.skip())
        self.rank_button.on_click(lambda _: self.toggle_ranking())
        self.stop_button.on_click(lambda _: self.toggle_stopped())
        self.view = widgets.VBox([
            self.status, self.save_status,
            widgets.HBox([self.choose_a, self.choose_b, self.both_button, self.back_button, self.skip_button, self.rank_button, self.stop_button],
                         layout=widgets.Layout(flex_flow="row wrap")),
            self.options, self.ranking,
        ])
        self.refresh()

    def refresh(self):
        self.back_button.disabled = not self.history
        if self.position >= len(self.comparisons):
            self.options.value = ""
            self.stop("All comparisons complete.")
            return
        model = fit_preferences(self)
        answered_count = model["answer_count"]
        shortlist = shortlist_status(self, model)
        if answered_count >= self.answer_budget:
            self.stop("Comparison budget reached. Review your shortlist, or Resume for 50 more answers.")
            return
        if shortlist["stable"] and answered_count >= self.stability_review_after:
            self.stop("The model's top-10 shortlist looks stable. Review it, or Resume to test further.")
            return
        self.status.value = (
            f'<h3 style="margin:8px 0;">{answered_count} answered · review budget {self.answer_budget}</h3>'
            '<p style="margin:0 0 12px;">Read both projects, then choose your preference. '
            'Learning from your choices and any supplied project tags. '
            'Points remain +10/−10; Both gives +10 each.</p>'
        )
        cards = []
        stats = preference_stats(self)
        for label, index in zip(("A", "B"), self.comparisons[self.position]):
            supervisor, title, info = self.projects[index - 1][:3]
            rating = self.reference_ratings[index - 1]
            reference = ", ".join(rating["ranks"])
            topics = "".join(
                '<span style="display:inline-block;border:1px solid #8884;border-radius:6px;'
                'padding:3px 8px;margin:0 6px 6px 0;font-size:12px;">'
                + html.escape(topic) + '</span>' for topic in rating["sections"]
            )
            tag_badges = "".join(metric_badge("", tag) for tag in rating["tags"])
            overview_parts = []
            if rating["summary"]:
                overview_parts.append('<h4>At a glance</h4><p>' + html.escape(rating["summary"]) + '</p>')
            if rating["potential_stack"]:
                overview_parts.append('<h4>Potential tools &amp; stack</h4><p>' + html.escape(rating["potential_stack"]) + '</p>')
            overview = "".join(overview_parts)
            preference = stats[index - 1]
            badges = metric_badge("Preference", f'{preference["score"]:.1f}/100 · {preference["answered"]} answers')
            if rating["ranks"]:
                badges += metric_badge("Interest", preference["group"]) + metric_badge("Draft ranking", reference)
            for field, metric_label in (("difficulty", "Difficulty"), ("ai_assistance", "AI assistance")):
                if rating[field] is not None:
                    badges += metric_badge(metric_label, f'{rating[field]}/10')
            supervisor_line = (
                f'<p style="margin:0 0 14px;">Supervisor: <b>{html.escape(supervisor.strip())}</b></p>'
                if supervisor.strip() else ""
            )
            if rating.get("draft_rank") is not None:
                badges += metric_badge("Draft rank", rating["draft_rank"])
            readable = rating.get("readable_description", "")
            description = format_description(info)
            if readable:
                description = (
                    render_readable_description(readable)
                    + '<details><summary>Original university description</summary>'
                    + description + '</details>'
                )
            cards.append(
                '<section style="flex:1 1 340px;min-width:0;border:1px solid #8886;'
                'border-radius:12px;padding:20px;box-sizing:border-box;">'
                f'<div style="font-size:14px;font-weight:400;">Option {label}</div>'
                f'<h3 style="font-size:20px;line-height:1.35;margin:8px 0 12px;">{html.escape(title.strip())}</h3>'
                f'<div aria-label="Topics" style="margin-bottom:6px;">{topics}</div>'
                f'{supervisor_line}'
                f'<div style="margin-bottom:12px;">{badges}</div>'
                '<div tabindex="0" aria-label="Project description" '
                'style="max-height:460px;overflow-y:auto;padding-right:12px;'
                'border-top:1px solid #8884;font-size:15px;line-height:1.65;overflow-wrap:anywhere;">'
                f'<div aria-label="Preference tags" style="margin-top:12px;">{tag_badges}</div>'
                f'{overview}{description}'
                '</div></section>'
            )
        self.options.value = (
            '<div style="display:flex;flex-wrap:wrap;gap:16px;margin-top:16px;">'
            + "".join(cards) + '</div>'
        )
        self.ranking.value = ""
        self.rank_button.description = "Show top 20"

    def choose(self, option):
        if self.choose_a.disabled:
            return
        if option not in (0, 1, 2):
            raise ValueError("Unknown choice")
        self.history.append((self.position, option))
        pair = self.comparisons[self.position]
        if option == 2:
            for index in pair:
                self.scores[index - 1] += 10
        else:
            self.scores[pair[option] - 1] += 10
            self.scores[pair[1 - option] - 1] -= 10
        self.position += 1
        prioritise_next(self)
        self.refresh()
        self.save_progress()

    def skip(self):
        if self.skip_button.disabled:
            return
        self.history.append((self.position, None))
        self.position += 1
        prioritise_next(self)
        self.refresh()
        self.save_progress()

    def back(self):
        if not self.history:
            return
        position, option = self.history.pop()
        if option is not None:
            pair = self.comparisons[position]
            if option == 2:
                for index in pair:
                    self.scores[index - 1] -= 10
            else:
                self.scores[pair[option] - 1] -= 10
                self.scores[pair[1 - option] - 1] += 10
        self.position = position
        self.choose_a.disabled = self.choose_b.disabled = self.both_button.disabled = False
        self.skip_button.disabled = self.stop_button.disabled = False
        self.stop_button.description = "Stop"
        self.refresh()
        self.save_progress()

    def toggle_ranking(self):
        if self.ranking.value:
            self.ranking.value = ""
            self.rank_button.description = "Show top 20"
        else:
            self.show_ranking()

    def show_ranking(self):
        stats = preference_stats(self)
        top = ranked_indices(self, stats)[:20]
        columns = []
        for start in (0, 10):
            rows = "".join(
                f'<li><b>{stats[i]["score"]:.1f}/100</b> — {html.escape(self.projects[i][1])}'
                f'<div style="font-size:12px;margin-bottom:8px;">{stats[i]["answered"]} answered appearances · '
                f'{html.escape(stats[i]["group"])} · {self.scores[i]} raw points</div></li>' for i in top[start:start + 10]
            )
            columns.append(f'<ol start="{start + 1}" style="margin:0;padding-left:28px;">{rows}</ol>')
        self.ranking.value = (
            '<h3>Current top 20</h3>'
            '<p style="font-size:13px;">Preferences learned from comparisons and optional shared tags. '
            'Project-specific evidence and shared tags both contribute; higher is better.</p>'
            '<div class="ranking-columns" style="display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:24px;">'
            + "".join(columns) + '</div>'
        )
        if any(stat["estimated_counts"] for stat in stats):
            self.ranking.value += '<p>Early appearance counts are estimated because this session predates complete undo history.</p>'
        self.rank_button.description = "Hide top 20"

    def toggle_stopped(self):
        if self.choose_a.disabled:
            if self.position >= len(self.comparisons):
                return
            answered_count = fit_preferences(self)["answer_count"]
            if answered_count >= self.answer_budget:
                self.answer_budget = answered_count + 50
            self.stability_review_after = answered_count + 25
            self.choose_a.disabled = self.choose_b.disabled = self.both_button.disabled = False
            self.skip_button.disabled = False
            self.stop_button.description = "Stop"
            self.stop_button.disabled = False
            self.refresh()
        else:
            self.stop()
        self.save_progress()

    def save_progress(self):
        if self.save_path is None:
            return
        try:
            save_session(self.save_path, self)
        except (OSError, ValueError) as error:
            self.save_status.value = (
                '<p role="alert"><b>Save failed — keep this kernel open.</b> '
                + html.escape(str(error)) + '</p>'
            )
        else:
            self.save_status.value = (
                f'<p style="margin:0;font-size:12px;">Saved locally · '
                f'{self.position} comparisons processed</p>'
            )

    def stop(self, message="Stopped. Your current scores are preserved."):
        self.choose_a.disabled = self.choose_b.disabled = self.both_button.disabled = True
        self.stop_button.description = "Resume"
        self.stop_button.disabled = self.position >= len(self.comparisons)
        self.skip_button.disabled = True
        self.back_button.disabled = not self.history
        self.status.value = f"<b>{message}</b>"
        self.show_ranking()


# Rerunning setup/display cells preserves the existing session.
previous_ranker = globals().get("ranker")
preserve_session = (
    previous_ranker is not None
    and previous_ranker.projects == data
    and Path(previous_ranker.save_path).resolve() == Path(save_path).resolve()
)
if preserve_session:
    # The active session owns its scores; a rerun may have replaced the global list.
    scores = previous_ranker.scores
position = previous_ranker.position if preserve_session else 0
comparisons = previous_ranker.comparisons if preserve_session else checks
history = getattr(previous_ranker, "history", []) if preserve_session else []
was_stopped = preserve_session and previous_ranker.choose_a.disabled
answer_budget = getattr(previous_ranker, "answer_budget", 250) if preserve_session else 250
stability_review_after = getattr(previous_ranker, "stability_review_after", 0) if preserve_session else 0
restored_adaptive = False
if not preserve_session:
    saved_session = load_session(save_path, data)
    if saved_session is not None:
        scores = saved_session["scores"]
        position = saved_session["position"]
        comparisons = [tuple(pair) for pair in saved_session["comparisons"]]
        history = [tuple(entry) for entry in saved_session["history"]]
        was_stopped = saved_session["stopped"]
        restored_adaptive = saved_session.get("adaptive_schedule", False)
        answer_budget = saved_session.get("answer_budget", 250)
        stability_review_after = saved_session.get("stability_review_after", 0)
        print(f"Restored saved session: {position} comparisons processed.")
if previous_ranker is not None:
    previous_ranker.stop()
    previous_ranker.view.close()
ranker = ProjectRanker(data, comparisons, scores, project_ratings, position=position, history=history, save_path=save_path, answer_budget=answer_budget, stability_review_after=stability_review_after)
# Preserve the current displayed pair when refreshing an already adaptive session.
if not restored_adaptive and (not preserve_session or not getattr(previous_ranker, "adaptive_schedule", False)):
    prioritise_next(ranker)
    ranker.refresh()
ranker.adaptive_schedule = True
if was_stopped:
    ranker.stop()
ranker.save_progress()
# Keep an existing browser connected to the refreshed controls.
if "ranker_window" in globals():
    with ranker_window.lock:
        ranker_window.ranker = ranker
display(ranker.view)
if preserve_session:
    print(f"Session restored: {position} comparisons processed; scores and undo history retained.")


In [ ]:
scores

In [ ]:
labled_scores = list(zip(range(1,height+1,1), scores))
labled_scores

In [ ]:
preference_scores = preference_stats(ranker)
labled_scores_sorted = [
    (index + 1, preference_scores[index]["score"])
    for index in ranked_indices(ranker, preference_scores)
]
labled_scores_sorted

In [ ]:
# Run only this cell to open a separate GUI without resetting your choices.
import importlib
import ranker_window as window_module
importlib.reload(window_module)
from ranker_window import RankerWindow

if "ranker" not in globals():
    raise RuntimeError("Run the notebook setup and display cells first.")
if "ranker_window" in globals():
    ranker_window.close()
ranker_window = RankerWindow(ranker)
ranker_window.open()
print(f"Dissertation Project Ranker window: {ranker_window.url}")
print("Keep this kernel running. Your existing scores and progress are preserved.")
